Live IDS prototype

This notebook prototypes and tests real-time intrusion detection by completing the following:

1. Capture network traffic
2. Convert captured PCAP files to Flow features (CSV)
3. Load the trained ML model
4. Score the flows
5. Generate alerts

Once completed this notebooke will become a standalone py file

In [2]:
#Comparing columns between the CICIDS2017 and a live sample captured using FlowMeter 

import pandas as pd

cic = pd.read_csv(r"E:\\Project Portfolio\\Dissertation\\Final-Year-IDS\\data\\CICIDS2017\\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
cic.columns = cic.columns.str.strip()

FlowM = pd.read_csv(r"E:\\Project Portfolio\\Dissertation\\Final-Year-IDS\\data\\example.pcap_Flow.csv")
FlowM.columns = FlowM.columns.str.strip()

cic_cols = set(cic.columns)
FlowM_cols = set(FlowM.columns)

print("CIC feature count:", len(cic_cols))
print("Live feature count:", len(FlowM_cols))
print("Shared features:", len(cic_cols & FlowM_cols))

print("\nMissing from live (sample):")
print(list((cic_cols - FlowM_cols))[:20])

print("\nExtra in live (sample):")
print(list((FlowM_cols - cic_cols))[:20])

CIC feature count: 79
Live feature count: 84
Shared features: 28

Missing from live (sample):
['Fwd Header Length', 'Avg Bwd Segment Size', 'Fwd IAT Total', 'FIN Flag Count', 'Total Fwd Packets', 'Bwd Packet Length Mean', 'Subflow Bwd Bytes', 'URG Flag Count', 'Avg Fwd Segment Size', 'Bwd Packet Length Min', 'Bwd Packets/s', 'ACK Flag Count', 'Fwd Avg Bulk Rate', 'Init_Win_bytes_forward', 'act_data_pkt_fwd', 'Fwd Avg Bytes/Bulk', 'Bwd Packet Length Max', 'Fwd Packet Length Min', 'Average Packet Size', 'Packet Length Variance']

Extra in live (sample):
['Pkt Len Std', 'Fwd Byts/b Avg', 'RST Flag Cnt', 'Pkt Len Mean', 'Bwd Pkt Len Mean', 'Flow Byts/s', 'Fwd Pkt Len Std', 'Tot Fwd Pkts', 'Subflow Bwd Pkts', 'ACK Flag Cnt', 'Bwd Pkt Len Std', 'Fwd Header Len', 'URG Flag Cnt', 'Bwd Blk Rate Avg', 'ECE Flag Cnt', 'Bwd Seg Size Avg', 'Bwd IAT Tot', 'Tot Bwd Pkts', 'TotLen Bwd Pkts', 'Pkt Len Var']


In [3]:
# Rename map to allow for easier future allignment and implement a function that will normalise features
# For both data types
rename_map = {
    "Pkt Len Mean": "Packet Length Mean",
    "Pkt Len Std": "Packet Length Std",
    "Pkt Len Max": "Packet Length Max",
    "Pkt Len Min": "Packet Length Min",

    "TotLen Fwd Pkts": "Total Length of Fwd Packets",
    "TotLen Bwd Pkts": "Total Length of Bwd Packets",

    "Tot Fwd Pkts": "Total Fwd Packets",
    "Tot Bwd Pkts": "Total Backward Packets",

    "Fwd Pkt Len Mean": "Fwd Packet Length Mean",
    "Fwd Pkt Len Std": "Fwd Packet Length Std",
    "Fwd Pkt Len Max": "Fwd Packet Length Max",
    "Fwd Pkt Len Min": "Fwd Packet Length Min",

    "PSH Flag Cnt": "PSH Flag Count",
    "ACK Flag Cnt": "ACK Flag Count",
    "RST Flag Cnt": "RST Flag Count",
    "URG Flag Cnt": "URG Flag Count",
    "SYN Flag Cnt": "SYN Flag Count",

    "Init Fwd Win Byts": "Init_Win_bytes_forward",
    "Init Bwd Win Byts": "Init_Win_bytes_backward",

    "Subflow Fwd Pkts": "Subflow Fwd Packets",
    "Subflow Bwd Pkts": "Subflow Bwd Packets",

    "Subflow Fwd Byts": "Subflow Fwd Bytes",
    "Subflow Bwd Byts": "Subflow Bwd Bytes",

    "Bwd Seg Size Avg": "Avg Bwd Segment Size",
    "Fwd Seg Size Avg": "Avg Fwd Segment Size",
}

def normalise_live_features(df, train_features):
    df = df.rename(columns=rename_map)
    df = df.reindex(columns=train_features, fill_value=0)

    return df

In [4]:
#Loading the previously saved bundel which includes: model, features, threshold, version and creation date

import joblib

bundle_path = r"E:\Project Portfolio\Dissertation\Final-Year-IDS\models\IDS_RF_v1.0"

bundle = joblib.load(bundle_path)
model = bundle["model"]
train_features = bundle["features"]
threshold = float(bundle["threshold"])

print("Loaded Features:", len(train_features))
print("Threshold:", threshold)

Loaded Features: 78
Threshold: 0.1


In [5]:
#normalising the live csv sample

X_norm = normalise_live_features(FlowM, train_features)

print("Normalised shape:", X_norm.shape)
print("Columns Identical? ", list(X_norm.columns) == list(train_features))

Normalised shape: (12, 78)
Columns Identical?  True


In [6]:
# Count how many features ended up all-zero (means missing/unmatched)
all_zero_cols = (X_norm.sum(axis=0) == 0).sum()
print("All-zero columns:", all_zero_cols, "out of", X_norm.shape[1])

# Check for NaN/inf after normalization
import numpy as np
print("Any NaN:", X_norm.isna().any().any())
print("Any inf:", np.isinf(X_norm.to_numpy()).any())

All-zero columns: 42 out of 78
Any NaN: False
Any inf: False


In [7]:
#testing model predicition

import numpy as np

proba = model.predict_proba(X_norm)[:, 1]
pred = (proba >= threshold).astype(int)

print("Rows:", len(pred))
print("Flagged malicious:", int(pred.sum()))
print("Max proba:", float(np.max(proba)))
print("Mean proba:", float(np.mean(proba)))

Rows: 12
Flagged malicious: 6
Max proba: 0.24705416624108578
Mean proba: 0.12421998767587317


In [8]:
# Function to score the file and return the probability of a malicious packet and the prediction
def score_file(csv_path, model, train_features, threshold):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    X = normalise_live_features(df, train_features)

    proba = model.predict_proba(X)[:,1]
    pred = (proba >= threshold).astype(int)

    results = df.copy()
    results["malicious_prob"] = proba
    results["Prediction"] = pred

    return results

#confirmed safe pcap
sample_file = r"E:\\Project Portfolio\\Dissertation\\Final-Year-IDS\\data\\example.pcap_Flow.csv"

results = score_file(sample_file, model, train_features, threshold)

results.head()

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,malicious_prob,Prediction
0,172.66.148.140-192.168.0.24-443-61421-6,192.168.0.24,61421,172.66.148.140,443,6,03/03/2026 05:07:29 pm,18684,0,2,...,0,0,0,0,0,0,0,No Label,0.079711,0
1,172.66.148.140-192.168.0.24-443-61422-6,192.168.0.24,61422,172.66.148.140,443,6,03/03/2026 05:07:29 pm,19852,0,2,...,0,0,0,0,0,0,0,No Label,0.079599,0
2,172.66.148.140-192.168.0.24-443-61420-6,192.168.0.24,61420,172.66.148.140,443,6,03/03/2026 05:07:29 pm,20534,0,2,...,0,0,0,0,0,0,0,No Label,0.077599,0
3,172.66.148.140-192.168.0.24-443-61419-6,192.168.0.24,61419,172.66.148.140,443,6,03/03/2026 05:07:29 pm,21232,0,2,...,0,0,0,0,0,0,0,No Label,0.077599,0
4,192.168.0.24-63.176.195.25-63938-443-6,192.168.0.24,63938,63.176.195.25,443,6,03/03/2026 05:07:29 pm,34515,1,2,...,0,0,0,0,0,0,0,No Label,0.104187,1


--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [55]:
import os, time, subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

# --- Paths ---
root = Path(r"E:\Project Portfolio\Dissertation\Final-Year-IDS")
pcap_dir = root/"live"/"pcaps"
flow_dir = root/"live"/"flows"
alert_dir = root/"live"/"alerts"
alert_csv = alert_dir/"alerts.csv"

#Iterate through the array and make a the new directory based on the path variables
for d in [pcap_dir, flow_dir, alert_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Model Bundel already loaded (Cell 3)
print("Loaded model. Features:", len(train_features), "Threshold:", threshold)

# CICFlowMeter batch file
cicFlowBat = Path(r"C:\Tools\CICFlowMeter-4.0\bin\cfm.bat")
assert cicFlowBat.exists(), f"Cannot find CICFlowMeter bat: {cicFlowBat}"

Loaded model. Features: 78 Threshold: 0.1


In [10]:
result = subprocess.run(["dumpcap", "-D"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

1. \Device\NPF_{F66FFFE5-46F1-4EEA-8319-A5EAEDC78D15} (Local Area Connection* 8)
2. \Device\NPF_{C60C8BEA-136D-4E70-A1AA-DE950E3B5223} (Local Area Connection* 7)
3. \Device\NPF_{4E4E80C3-1B1F-46BC-918A-2465B94FD857} (Local Area Connection* 6)
4. \Device\NPF_{3350E384-C38F-4CD7-BA41-0D2F80A2E852} (Bluetooth Network Connection 3)
5. \Device\NPF_{D0E67351-138D-4A72-BB05-C199700D45FF} (WiFi)
6. \Device\NPF_{3A20BF10-E273-4B83-92A0-F3C12A05A24D} (Local Area Connection* 10)
7. \Device\NPF_{BCCF7766-11DF-4187-9176-4972769148DE} (Local Area Connection* 9)
8. \Device\NPF_Loopback (Adapter for loopback traffic capture)
9. \Device\NPF_{174CE4F1-39CB-4EAC-805B-2EA6E610D709} (Ethernet)




In [62]:
#Select the desired interface (In this case WiFi)
interface = "5"
rotate_time = 10 #pcap duration per file

capture_cmd = [
    "dumpcap",
    "-i", interface,
    "-b", f"duration:{rotate_time}",
    "-w", str(pcap_dir / "capture.pcap")
]

print("Starting capture:", " ".join(capture_cmd))
capture_proc = subprocess.Popen(capture_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print("Capture PID:", capture_proc.pid)

Starting capture: dumpcap -i 5 -b duration:10 -w E:\Project Portfolio\Dissertation\Final-Year-IDS\live\pcaps\capture.pcap
Capture PID: 26524


In [ ]:
# Function to convert pcap to csv using CICFlowMeter
def pcap_to_csv(pcap_path: Path) -> Path:
    output = flow_dir
    cmd = ["cmd", "/c",".\\cfm.bat", str(pcap_path), str(output)]
    completed = subprocess.run(
        cmd,
        cwd=cicFlowBat.parent,   # run inside CICFlowMeter/bin
        capture_output=True,
        text=True
    )

    if completed.returncode != 0:
        raise RuntimeError(
            f"CICFlowMeter failed for {pcap_path.name}\n"
            f"stdout:\n{completed.stdout}\n"
            f"stderr:\n{completed.stderr}"
        )

    if not output.exists():
        raise FileNotFoundError(
            f"Expected CSV not created: {output}\n"
            f"stdout:\n{completed.stdout}\n"
            f"stderr:\n{completed.stderr}"
        )

    return output

In [ ]:
# Function that will score the csv using the trained IDS model and return a dataframe containing flow metadata for readability
# malicious probability, prediction, severity level and an alert message

def score_csv(flow_csv: Path) -> pd.DataFrame:

    # Load and define metadata columns
    df = pd.read_csv(flow_csv)
    df.columns = df.columns.str.strip()

    meta_headers = ["Flow ID", "Src IP", "Dst IP", "Src Port", "Dst Port", "Protocol", "Timestamp"]
    meta_check = [c for c in meta_headers if c in df.columns]
    metadata = df[meta_check].copy()

    # Prepare model features
    X = normalise_live_features(df, train_features)

    # Ensure features are matched
    if list(X.columns) != list(train_features):
        raise ValueError("Feature mismatch after normalisation")

    # Prediction
    chance = model.predict_proba(X)[:, 1]
    prediction = (chance >= threshold).astype(int)

    # Severity level definement
    def severity(p):
        if p > 0.80:
            return "CRITICAL"
        elif p > 0.50:
            return "HIGH"
        elif p > 0.25:
            return "MEDIUM"
        elif p > threshold:
            return "LOW"
        else:
            return "SAFE"
        
    #Alert Messages
    alerts = []

    for i, row in metadata.iterrows():
        if prediction[i] == 1:
            msg = (
                f"⚠️ Potential malicious packet detected | "
                f"{row.get('Src IP', '?')}:{row.get('Src Port', '?')} -> "
                f"{row.get('Dst IP', '?')}:{row.get('Dst Port', '?')} | "
                f"Protocol: {row.get('Protocol', '?')} | "
                f"Probability: {chance[i]:.3f}"
            )
        else:
            msg = ""
        alerts.append(msg)

    # Output Dataframe
    results = metadata.copy()
    results["malicious_prob"] = chance
    results["prediction"] = prediction
    results["severity"] = severity
    results["alert"] = alerts

    return results


In [ ]:
# Convert pcap to csv file
flow_pcap = Path(r"E:\Project Portfolio\Dissertation\Final-Year-IDS\live\pcaps\capture_00001_20260305165920.pcap")
output_folder = pcap_to_csv(flow_pcap)
print("CICFlowMeter output folder:", output_folder)

#Fid the CSV file created
csv_file = flow_dir / f"{flow_pcap.stem}.pcap_Flow.csv"
fallback_csv = sorted(flow_dir.glob(f"{flow_pcap.stem}*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)


# Score the csv file using the IDS model


CSV created:  E:\Project Portfolio\Dissertation\Final-Year-IDS\live\flows


In [63]:
capture_proc.terminate()
print("Capture stopped.")

Capture stopped.
